In [1]:
import polars as pl

from ohlc_dss_model.data import load_parquet, remove_incomplete_days, intraday_session_tagging, session_tagging, filter_valid_sessions

from ohlc_dss_model.features import (
    aggregate_sessions, yang_zhang, 
    PRE_NY_SPEC, FULL_DAY_SPEC, 
    calculate_excursion_bands, assign_direction, 
    detect_pivots, pivot_extraction,
    build_fred_macro, encode_news_context, build_individual_event_flags
)

from ohlc_dss_model.utils import convert_to_timezone, plot_session

from ohlc_dss_model.config import config

from pathlib import Path

from datetime import date

import os
from dotenv import load_dotenv

In [2]:
START_DATE = date(2011, 4, 5)

In [3]:
def load_raw_data(file_path: Path) -> pl.DataFrame:
    raw_data = load_parquet(file_path)
    raw_data = convert_to_timezone(raw_data)
    raw_data = session_tagging(raw_data)
    raw_data = intraday_session_tagging(raw_data)
    raw_data = remove_incomplete_days(raw_data)
    return raw_data.select(pl.col([
        "DateTime", "Session", "Intraday_Session", 
        "Open", "High", "Low", "Close", "Volume"
    ]))

In [4]:
raw_30min_data = load_raw_data(config.data.raw_folder_path / "nq_30m.parquet")
raw_1min_data = load_raw_data(config.data.raw_folder_path / "nq_1m.parquet")

aggregated_data = aggregate_sessions(raw_30min_data)
aggregated_data = filter_valid_sessions(aggregated_data)

aggregated_data = aggregated_data.with_columns(
    pl.col("O_Pre_Target_1").alias("O_Ref")
)

aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

aggregated_data = assign_direction(aggregated_data)
aggregated_data = calculate_excursion_bands(aggregated_data)


In [5]:
print(raw_30min_data.head(3))

shape: (3, 8)
┌──────────────────┬────────────┬──────────────────┬─────────┬─────────┬─────────┬────────┬────────┐
│ DateTime         ┆ Session    ┆ Intraday_Session ┆ Open    ┆ High    ┆ Low     ┆ Close  ┆ Volume │
│ ---              ┆ ---        ┆ ---              ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---    │
│ datetime[ns, Ame ┆ date       ┆ str              ┆ f64     ┆ f64     ┆ f64     ┆ f64    ┆ u64    │
│ rica/New_York]   ┆            ┆                  ┆         ┆         ┆         ┆        ┆        │
╞══════════════════╪════════════╪══════════════════╪═════════╪═════════╪═════════╪════════╪════════╡
│ 2011-01-02       ┆ 2011-01-03 ┆ Pre_Target_1     ┆ 2220.0  ┆ 2223.75 ┆ 2219.25 ┆ 2220.5 ┆ 889    │
│ 18:00:00 EST     ┆            ┆                  ┆         ┆         ┆         ┆        ┆        │
│ 2011-01-02       ┆ 2011-01-03 ┆ Pre_Target_1     ┆ 2220.0  ┆ 2222.0  ┆ 2219.75 ┆ 2221.5 ┆ 238    │
│ 18:30:00 EST     ┆            ┆                  ┆         ┆         ┆     

In [6]:
print(raw_1min_data.head(3))

shape: (3, 8)
┌──────────────────┬────────────┬──────────────────┬─────────┬─────────┬────────┬─────────┬────────┐
│ DateTime         ┆ Session    ┆ Intraday_Session ┆ Open    ┆ High    ┆ Low    ┆ Close   ┆ Volume │
│ ---              ┆ ---        ┆ ---              ┆ ---     ┆ ---     ┆ ---    ┆ ---     ┆ ---    │
│ datetime[ns, Ame ┆ date       ┆ str              ┆ f64     ┆ f64     ┆ f64    ┆ f64     ┆ u64    │
│ rica/New_York]   ┆            ┆                  ┆         ┆         ┆        ┆         ┆        │
╞══════════════════╪════════════╪══════════════════╪═════════╪═════════╪════════╪═════════╪════════╡
│ 2011-01-02       ┆ 2011-01-03 ┆ Pre_Target_1     ┆ 2220.0  ┆ 2223.0  ┆ 2220.0 ┆ 2222.75 ┆ 169    │
│ 18:00:00 EST     ┆            ┆                  ┆         ┆         ┆        ┆         ┆        │
│ 2011-01-02       ┆ 2011-01-03 ┆ Pre_Target_1     ┆ 2222.0  ┆ 2222.75 ┆ 2222.0 ┆ 2222.25 ┆ 10     │
│ 18:01:00 EST     ┆            ┆                  ┆         ┆         ┆     

In [7]:
print(aggregated_data.drop_nulls().head(5))

shape: (5, 37)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Session   ┆ O_Target_ ┆ O_Pre_Tar ┆ O_Pre_Tar ┆ … ┆ Band_FE_N ┆ Band_FE_N ┆ Band_FE_P ┆ Band_FE_ │
│ ---       ┆ 2         ┆ get_2     ┆ get_1     ┆   ┆ eg_Upper  ┆ eg_Lower  ┆ os_Upper  ┆ Pos_Lowe │
│ date      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ r        │
│           ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2011-01-2 ┆ 2293.5    ┆ 2274.0    ┆ 2268.75   ┆ … ┆ 2239.2161 ┆ 2232.7434 ┆ 2304.7565 ┆ 2298.283 │
│ 4         ┆           ┆           ┆           ┆   ┆ 9         ┆ 71        ┆ 29        ┆ 81       │
│ 2011-01-2 ┆ 2295.5    ┆ 2297.75   ┆ 2296.75   ┆ … ┆ 2264.9987 ┆ 2257.9265 

In [8]:
print(aggregated_data.columns)

['Session', 'O_Target_2', 'O_Pre_Target_2', 'O_Pre_Target_1', 'O_Target_1', 'H_Target_2', 'H_Pre_Target_2', 'H_Pre_Target_1', 'H_Target_1', 'L_Target_2', 'L_Pre_Target_2', 'L_Pre_Target_1', 'L_Target_1', 'C_Target_2', 'C_Pre_Target_2', 'C_Pre_Target_1', 'C_Target_1', 'O_Ref', 'Sigma_Historical', 'Sigma_Today', 'Z_Body', 'Z_Sigma', 'Tau', 'Direction', '_delta_t', 'Band_AE_Pos_Center', 'Band_AE_Neg_Center', 'Band_FE_Pos_Center', 'Band_FE_Neg_Center', 'Band_AE_Neg_Upper', 'Band_AE_Neg_Lower', 'Band_AE_Pos_Upper', 'Band_AE_Pos_Lower', 'Band_FE_Neg_Upper', 'Band_FE_Neg_Lower', 'Band_FE_Pos_Upper', 'Band_FE_Pos_Lower']


***
### **XGBoost Volume-Based Tabular Multiasset Approach**

#### **Multi-Asset Macros**

In [20]:
# This is where target_1 approach and target_2 pipeline diverge we will later combine it in the end

In [9]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
END_DATE = aggregated_data.select(pl.col("Session").max()).item()
print(END_DATE)

2026-04-16


In [10]:
# aggregated_data = build_fred_macro(aggregated_data, FRED_API_KEY, START_DATE, END_DATE)
aggregated_data = load_parquet(config.data.raw_folder_path / "cache" / "fred_load.parquet")

In [11]:
# from ohlc_dss_model.data import write_parquet
# write_parquet(aggregated_data, "fred_load", config.data.raw_folder_path / "cache")

In [12]:
print(aggregated_data.filter(pl.col("Session") >= START_DATE).select(['Session', 'Join_date',
        'vix_t1', 'us10y_t1', 'us2y_t1', 'effr_t1', 
        '10y_2y_spread_t1', 
        'vix_5d_delta', 'us10y_5d_delta', 
        'vix_pct_rank_1y_t1']).drop_nulls().head(3))

shape: (3, 10)
┌────────────┬────────────┬────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ Session    ┆ Join_date  ┆ vix_t1 ┆ us10y_t1 ┆ … ┆ 10y_2y_spr ┆ vix_5d_de ┆ us10y_5d_ ┆ vix_pct_r │
│ ---        ┆ ---        ┆ ---    ┆ ---      ┆   ┆ ead_t1     ┆ lta       ┆ delta     ┆ ank_1y_t1 │
│ date       ┆ date       ┆ f64    ┆ f64      ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆        ┆          ┆   ┆ f64        ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2011-04-05 ┆ 2011-04-05 ┆ 17.5   ┆ 3.45     ┆ … ┆ 2.68       ┆ -1.94     ┆ -0.02     ┆ 0.188889  │
│ 2011-04-06 ┆ 2011-04-06 ┆ 17.25  ┆ 3.5      ┆ … ┆ 2.66       ┆ -0.91     ┆ 0.0       ┆ 0.143646  │
│ 2011-04-07 ┆ 2011-04-07 ┆ 16.9   ┆ 3.56     ┆ … ┆ 2.71       ┆ -0.81     ┆ 0.09      ┆ 0.131868  │
└────────────┴────────────┴────────┴──────────┴───┴────────────┴───────────┴

In [13]:
print(aggregated_data.select(['Session', 'Join_date',
        'vix_t1', 'us10y_t1', 'us2y_t1', 'effr_t1', 
        '10y_2y_spread_t1', 
        'vix_5d_delta', 'us10y_5d_delta', 
        'vix_pct_rank_1y_t1']).drop_nulls().tail(2))

shape: (2, 10)
┌────────────┬────────────┬────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ Session    ┆ Join_date  ┆ vix_t1 ┆ us10y_t1 ┆ … ┆ 10y_2y_spr ┆ vix_5d_de ┆ us10y_5d_ ┆ vix_pct_r │
│ ---        ┆ ---        ┆ ---    ┆ ---      ┆   ┆ ead_t1     ┆ lta       ┆ delta     ┆ ank_1y_t1 │
│ date       ┆ date       ┆ f64    ┆ f64      ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆        ┆          ┆   ┆ f64        ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-15 ┆ 2026-04-15 ┆ 18.36  ┆ 4.26     ┆ … ┆ 0.5        ┆ -7.42     ┆ -0.07     ┆ 0.604457  │
│ 2026-04-16 ┆ 2026-04-16 ┆ 18.17  ┆ 4.29     ┆ … ┆ 0.53       ┆ -2.87     ┆ 0.0       ┆ 0.590529  │
└────────────┴────────────┴────────┴──────────┴───┴────────────┴───────────┴───────────┴───────────┘


In [14]:
event_table = load_parquet(config.data.processed_folder_path / "event_table.parquet")
aggregated_data = encode_news_context(aggregated_data, event_table)

In [15]:
aggregated_data.filter(pl.col("Session") >= START_DATE).select(["Session", "e_yesterday", "e_today", "e_tomorrow"]).drop_nulls().head(5)

Session,e_yesterday,e_today,e_tomorrow
date,i8,i8,i8
2011-04-05,0,1,0
2011-04-06,1,0,2
2011-04-07,0,2,0
2011-04-08,2,0,0
2011-04-11,0,0,0


In [16]:
# aggregated_data = build_individual_event_flags(aggregated_data, FRED_API_KEY, START_DATE, date(2026, 4, 30))
aggregated_data = load_parquet(config.data.raw_folder_path / "cache" / "fomc_load.parquet")

In [17]:
aggregated_data.filter(pl.col("is_fomc_day") == True).select(["Session"]).drop_nulls().tail(5)

Session
date
2025-09-17
2025-10-29
2025-12-10
2026-01-28
2026-03-18


In [18]:
# from ohlc_dss_model.data import write_parquet
# write_parquet(aggregated_data, "fomc_load", config.data.raw_folder_path / "cache")

In [19]:
aggregated_data.tail(5)

Session,O_Pre_Target_2,O_Pre_Target_1,O_Target_1,O_Target_2,H_Pre_Target_2,H_Pre_Target_1,H_Target_1,H_Target_2,L_Pre_Target_2,L_Pre_Target_1,L_Target_1,L_Target_2,C_Pre_Target_2,C_Pre_Target_1,C_Target_1,C_Target_2,O_Ref,Sigma_Historical,Sigma_Today,Z_Body,Z_Sigma,Tau,Direction,_delta_t,Band_AE_Pos_Center,Band_AE_Neg_Center,Band_FE_Pos_Center,Band_FE_Neg_Center,Band_AE_Neg_Upper,Band_AE_Neg_Lower,Band_AE_Pos_Upper,Band_AE_Pos_Lower,Band_FE_Neg_Upper,Band_FE_Neg_Lower,Band_FE_Pos_Upper,Band_FE_Pos_Lower,vix_t1,us10y_t1,us2y_t1,effr_t1,10y_2y_spread_t1,vix_5d_delta,us10y_5d_delta,vix_pct_rank_1y_t1,Join_date,e_today,e_yesterday,e_tomorrow,is_fomc_day,is_fomc_week,days_to_fomc,is_nfp_day,is_cpi_day,is_core_cpi_day
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,i8,i8,i8,bool,bool,i16,bool,bool,bool
2026-04-10,25229.25,25237.0,25324.5,25265.5,25386.0,25303.75,25393.0,25338.0,25202.75,25193.0,25244.75,25223.25,25336.25,25228.25,25264.5,25333.0,25237.0,0.015757,0.006078,0.240953,0.691766,0.480928,"""neutral""",53.69565,25448.86126,25025.13874,25716.320867,24757.679133,25078.83439,24971.44309,25502.55691,25395.16561,24811.374783,24703.983482,25770.016518,25662.625217,19.49,4.29,3.78,3.64,0.51,-4.38,-0.02,0.688022,2026-04-10,3,2,0,false,false,19,false,true,true
2026-04-13,25115.5,24980.0,25125.75,25411.0,25194.75,25119.0,25360.75,25599.25,25073.5,24904.5,25120.25,25396.0,25124.75,25115.5,25309.5,25585.5,24980.0,0.017534,0.016756,1.365905,0.797482,0.447919,"""bullish""",39.361303,25142.484743,24817.515257,25266.801046,24693.198954,24856.87656,24778.153954,25181.846046,25103.12344,24732.560256,24653.837651,25306.162349,25227.439744,19.23,4.31,3.81,3.64,0.5,-4.38,-0.04,0.67688,2026-04-11,0,0,1,false,false,16,false,false,false
2026-04-14,25586.0,25575.25,25692.75,25914.25,25701.0,25619.25,25899.75,26003.25,25581.5,25560.0,25651.75,25884.25,25694.5,25584.75,25875.75,25990.0,25575.25,0.018336,0.005116,0.877334,0.862704,0.430655,"""bullish""",44.844586,25765.852443,25384.647557,25932.110218,25218.389782,25429.492143,25339.802971,25810.697029,25721.007857,25263.234368,25173.545196,25976.954804,25887.265632,19.12,4.3,3.78,3.64,0.52,-5.05,-0.04,0.668524,2026-04-14,1,0,0,false,false,15,false,false,false
2026-04-15,25988.5,25985.75,25974.5,26144.25,26048.0,26058.0,26215.5,26377.5,25933.75,25966.0,25970.0,26136.75,25975.0,25988.0,26196.75,26359.5,25985.75,0.018314,0.003498,0.779765,0.889093,0.424215,"""bullish""",47.64735,26128.518606,25842.981394,26361.341152,25610.158848,25890.628744,25795.334044,26176.165956,26080.871256,25657.806198,25562.511497,26408.988503,26313.693802,18.36,4.26,3.76,3.64,0.5,-7.42,-0.07,0.604457,2026-04-15,0,1,2,false,false,14,false,false,false
2026-04-16,26479.75,26350.0,26427.75,26474.25,26488.0,26486.0,26543.0,26509.75,26364.5,26345.0,26275.5,26355.0,26425.5,26480.75,26533.25,26464.25,26350.0,0.017909,0.004608,0.241584,0.90692,0.420025,"""neutral""",48.25674,26487.952665,26212.047335,26742.43367,25957.56633,26260.304075,26163.790595,26536.209405,26439.695925,26005.823069,25909.30959,26790.69041,26694.176931,18.17,4.29,3.76,3.64,0.53,-2.87,0.0,0.590529,2026-04-16,2,0,0,false,false,13,false,false,false


***
#### **Band Interaction**

In [ ]:
aggregated_data = aggregated_data.with_columns([
    (pl.col("Sigma_Historical").shift(1) * pl.col("O_Ref")).alias("Sigma_Price"),

    ((pl.col("C_Pre_Target_2") - pl.col("Band_FE_Pos_Center")) / pl.col("Sigma_Price")).alias("delta_ps2_fe_pos_center"),
    ((pl.col("C_Pre_Target_2") - pl.col("Band_AE_Pos_Center")) / pl.col("Sigma_Price")).alias("delta_ps2_ae_pos_center"),
    ((pl.col("Band_FE_Neg_Center") - pl.col("C_Pre_Target_2")) / pl.col("Sigma_Price")).alias("delta_ps2_fe_neg_center"),
    ((pl.col("Band_AE_Neg_Center") - pl.col("C_Pre_Target_2")) / pl.col("Sigma_Price")).alias("delta_ps2_ae_neg_center"),

    
])